# 37 — Is the production posterior right *for this window*? L-C2ST (System I, N3)

SBC and TARP test the posterior on average over the prior; the coverage tables test it per scenario. They cannot say whether the posterior at one particular observation is correct. The **local classifier two-sample test (L-C2ST**, Linhart et al. 2023;
`sbi.diagnostics.LC2ST`) can: a classifier is trained to tell apart (θ from the prior, x) from (θ from the posterior, x) on a calibration set; at a given x_o, the classifier's accuracy on posterior draws θ_o ~ q(θ | x_o) is compared with its distribution under the
null (a posterior that is exactly right). A small p-value says that the posterior at x_o differs from the truth in a way the classifier can see. This replaces the NUTS ground truth (decision 2026-11-09; the diagnosis is in the plan, SI-1).

**Settings (fixed before running; runbook §2f row e2).** sbi 0.26.1 `LC2ST` with its **defaults**: sklearn MLPClassifier with hidden layers (10·dim θ, 10·dim θ) = (20, 20), adam, early stopping, 1 fold, 1 classifier, seed 1, 100 null trials, permutation null, no z-scoring.
**Calibration set:** N_cal = 20,000 pairs (θ from the prior; x simulated under the production model: protocol P2, N3 sensors, true inlet from the notebook-35 distribution, *measured* Ti, Tci with ε = 0.1 K; new seeds 9 500 001–9 500 004) with one draw from the production ensemble (notebook 36) at each x.
**Test windows:** the first 5 windows of each scenario Sc1–Sc5 in D1 (25 windows, fixed by index), 2,000 ensemble draws per window as θ_o, x_o = the window's S1m features. **Reported:** the per-window p-value, the number below 0.05 (about 1.25 expected under a calibrated posterior) and the minimum; no decision rule.
**Positive controls (section 4; corrected after the first run showed the first design could not work):** the whole test is run for deliberately wrong posteriors (β shifted by +0.10, width halved), with the wrong posterior used for both the calibration draws and the test draws; and labelled variants with standardised inputs and with a compressed observation (section 4).


In [1]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent
for p in (ROOT / "src", ROOT / "scripts"):
    sys.path.insert(0, str(p))
os.environ["CSTR_SENSOR_MODEL"] = "N3"
warnings.filterwarnings("ignore")
import importlib, ws_eval, ws_phase3, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet
for _m in (ws_eval, ws_phase3, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet):
    importlib.reload(_m)
we, P3, M, P36, wi = ws_eval, ws_phase3, nb35_measured_inlet, nb36_production_measured_inlet, ws_inlet
from sbi.diagnostics import LC2ST
R = ROOT / "results"; OUTD = R / "37_N3"; OUTD.mkdir(exist_ok=True); T0 = time.time()
N_CAL, N_THETA_O, N_NULL = 20_000, 2_000, 100
post, prod = P3.load_production_posterior()
print("Production posterior:", prod["production_budget"], "simulations, ensemble of", len(prod["members"]))


Production posterior: 20000 simulations, ensemble of 5


## 1. Calibration set under the production model (cached)

In [2]:
f = OUTD / "cal.npz"
if f.exists():
    c = we.load_npz(f); th, X = c["theta"], c["x"]
else:
    th = we.prior_draws(N_CAL, 9_500_001); inl = M._inlets(N_CAL, 9_500_002)
    b29 = wi.simulate(th, inl, M.PROTOCOL, 9_500_003, features="S0_base29", **M.NOISE)
    X = P36.features(b29, inl, 9_500_004); we.save_npz(f, {"theta": th, "x": X, "inlet": inl})
assert th.shape == (N_CAL, 2) and X.shape == (N_CAL, 34) and np.isfinite(X).all()
smp, leak = we.sample_within_prior(post, X, 1, rng_seed=1)
print(f"calibration set {X.shape}; one ensemble draw per x (prior leakage {leak.mean():.4f}) | {time.time() - T0:.0f}s")
T_ = torch.tensor(th, dtype=torch.float32); X_ = torch.tensor(X, dtype=torch.float32); S_ = torch.tensor(smp[:, 0, :], dtype=torch.float32)


calibration set (20000, 34); one ensemble draw per x (prior leakage 0.0068) | 2s


## 2. Train the L-C2ST classifier and its null (sbi defaults)

In [3]:
t = time.time(); lc = LC2ST(T_, X_, S_, seed=1, num_trials_null=N_NULL)
lc.train_on_observed_data(); lc.train_under_null_hypothesis(verbosity=0)
print(f"classifier and {N_NULL} null classifiers trained in {time.time() - t:.0f}s")


classifier and 100 null classifiers trained in 283s


## 3. The 25 test windows: per-window p-values

In [4]:
d1 = P3.load_d1(); Xo_all = P3.production_features(d1); sid = d1["scenario_id"]
SCN = {1: "Sc1", 2: "Sc2", 3: "Sc3", 4: "Sc4", 5: "Sc5"}
wins = np.concatenate([np.where(sid == s)[0][:5] for s in SCN])
so, _ = we.sample_within_prior(post, Xo_all[wins], N_THETA_O, rng_seed=2)
def pvals(samples):
    return np.array([lc.p_value(torch.tensor(samples[k], dtype=torch.float32), torch.tensor(Xo_all[w], dtype=torch.float32)) for k, w in enumerate(wins)])
p_main = pvals(so)
res = pd.DataFrame(dict(window=wins, scenario=[SCN[int(sid[w])] for w in wins], alpha_true=d1["theta"][wins, 2], beta_true=d1["theta"][wins, 3],
                        post_mean_alpha=so.mean(1)[:, 0], post_mean_beta=so.mean(1)[:, 1], p_value=p_main))
display(res.round(3)); res.to_csv(R / "37_N3_lc2st_pvalues.csv", index=False)
from scipy import stats
print(f"p < 0.05 in {(p_main < 0.05).sum()} of {len(p_main)} windows (expected about {0.05 * len(p_main):.2f} under a calibrated posterior) | minimum p = {p_main.min():.3f} | "
      f"median p = {np.median(p_main):.3f} | KS test of the 25 p-values against U(0,1): p = {stats.kstest(p_main, 'uniform').pvalue:.3f}")
print("the null is estimated with 100 trials: a printed p = 0.00 means fewer than 1 in 100 null statistics reach the observed one")


,window,scenario,alpha_true,beta_true,post_mean_alpha,post_mean_beta,p_value
0,0,Sc1,1.00,1.00,0.958,0.920,0.69
1,1,Sc1,1.00,1.00,0.986,1.013,0.65
2,2,Sc1,1.00,1.00,1.020,1.028,0.53
3,3,Sc1,1.00,1.00,0.993,1.012,0.82
4,4,Sc1,1.00,1.00,0.982,1.035,0.39
5,50,Sc2,1.00,0.85,0.962,0.891,0.64
6,51,Sc2,1.00,0.85,1.041,0.850,0.62
7,52,Sc2,1.00,0.85,1.006,0.820,0.71
8,53,Sc2,1.00,0.85,1.035,0.835,0.74
9,54,Sc2,1.00,0.85,0.956,0.841,0.84


p < 0.05 in 0 of 25 windows (expected about 1.25 under a calibrated posterior) | minimum p = 0.160 | median p = 0.620 | KS test of the 25 p-values against U(0,1): p = 0.009
the null is estimated with 100 trials: a printed p = 0.00 means fewer than 1 in 100 null statistics reach the observed one


## 4. Positive controls, done correctly (added after the first run: the first power check was flawed)

The first version of the power check evaluated deliberately wrong posterior draws θ_o' with a classifier that had been trained on the *correct* posterior. That cannot work: a classifier trained to separate truth from a correct posterior has never seen the wrong one, so the shifted draws change nothing (the first run
rejected the production posterior and the shifted ones at the same rate, 0.52 and 0.52/0.48, exactly this symptom). **A valid positive control runs the whole L-C2ST for the wrong posterior q'**: the calibration posterior draws and the test draws both come from q'. Wrong posteriors: β shifted by +0.10 (2.3 posterior sd) and the width halved about the ensemble mean (applied to the one calibration draw per x and to the test draws).
Each variant (classifier input and size) is run with the production posterior and with the wrong ones, same 25 windows. **Variants (fixed before this run, after seeing that the pre-set default behaves oddly):** *default* (raw inputs, sbi default (20, 20), 100 null trials; the pre-set headline); *std* (inputs standardised with the calibration-set mean and sd, θ scaled to the prior, (20, 20), 100 null trials);
*comp* (std + x compressed to the ensemble's posterior mean and sd, 6 inputs, (20, 20), 100 null trials). A variant is *informative* if it flags the wrong posteriors in at least 80% of the windows and the production posterior in at most about 10% (the nominal 5% plus the sampling noise of 25 windows).

In [5]:
c_mu, c_sd = X.mean(0), X.std(0); tx = lambda x: (np.asarray(x) - c_mu) / c_sd; tt = lambda t_: (np.asarray(t_) - 0.8) / 0.23
dr, _ = we.sample_within_prior(post, X, 500, rng_seed=7); dro, _ = we.sample_within_prior(post, Xo_all[wins], 500, rng_seed=8)
comp = np.concatenate([dr.mean(1), dr.std(1)], axis=1); c2_mu, c2_sd = comp.mean(0), comp.std(0)
comp_cal = (comp - c2_mu) / c2_sd; comp_obs = (np.concatenate([dro.mean(1), dro.std(1)], axis=1) - c2_mu) / c2_sd
m_cal = dr.mean(1); m_obs = so.mean(1)                                   # ensemble means: the centre of the width control
def make_q(kind, S_cal, S_obs):
    if kind == "production posterior":
        return S_cal, S_obs
    if kind == "beta shifted +0.10":
        a, b = S_cal.copy(), S_obs.copy(); a[:, 1] += 0.10; b[:, :, 1] += 0.10; return a, b
    if kind == "width x0.5":
        return m_cal + 0.5 * (S_cal - m_cal), m_obs[:, None, :] + 0.5 * (S_obs - m_obs[:, None, :])
def run_variant(vname, kind):
    S_cal, S_obs = make_q(kind, smp[:, 0, :], so)
    if vname == "default":
        Tc, Xc, Sc = th, X, S_cal; to = lambda s: s; xo = lambda k: Xo_all[wins[k]]; kw = {}
    elif vname == "std":
        Tc, Xc, Sc = tt(th), tx(X), tt(S_cal); to = tt; xo = lambda k: tx(Xo_all[wins[k]]); kw = {}
    else:
        Tc, Xc, Sc = tt(th), comp_cal, tt(S_cal); to = tt; xo = lambda k: comp_obs[k]; kw = {}
    t0_ = time.time()
    lcv = LC2ST(torch.tensor(Tc, dtype=torch.float32), torch.tensor(Xc, dtype=torch.float32), torch.tensor(Sc, dtype=torch.float32), seed=1, num_trials_null=N_NULL, **kw)
    lcv.train_on_observed_data(); lcv.train_under_null_hypothesis(verbosity=0)
    p = np.array([lcv.p_value(torch.tensor(to(S_obs[k]), dtype=torch.float32), torch.tensor(xo(k), dtype=torch.float32)) for k in range(len(wins))])
    print(f"{vname} | {kind}: {time.time() - t0_:.0f}s | p<0.05 in {(p < 0.05).mean():.2f} of windows, median p {np.median(p):.2f}", flush=True)
    return p
RUNS = [("default", "beta shifted +0.10"), ("std", "production posterior"), ("std", "beta shifted +0.10"), ("std", "width x0.5"), ("comp", "production posterior"), ("comp", "beta shifted +0.10")]
pv_runs = {("default", "production posterior"): p_main}
for vn, kind in RUNS:
    pv_runs[(vn, kind)] = run_variant(vn, kind)
rows = [dict(variant=v, posterior=k, frac_p_below_0_05=float((p < 0.05).mean()), median_p=float(np.median(p)), min_p=float(p.min()), n_windows=len(p)) for (v, k), p in pv_runs.items()]
pc = pd.DataFrame(rows).sort_values(["variant", "posterior"]); display(pc.round(3)); pc.to_csv(R / "37_N3_lc2st_positive_controls.csv", index=False)
pw = pd.DataFrame({f"{v} | {k}": p for (v, k), p in pv_runs.items()}); pw.insert(0, "window", wins); pw.insert(1, "scenario", res.scenario.values)
display(pw.round(2)); pw.to_csv(R / "37_N3_lc2st_pvalues_all_variants.csv", index=False)
print("posterior sd of beta in these windows (mean):", round(float(so.std(1)[:, 1].mean()), 3), "| a +0.10 shift is", round(0.10 / float(so.std(1)[:, 1].mean()), 1), "posterior sd")
print(f"notebook wall time {(time.time() - T0) / 60:.1f} min")


default | beta shifted +0.10: 284s | p<0.05 in 0.92 of windows, median p 0.02


std | production posterior: 278s | p<0.05 in 0.00 of windows, median p 0.54


std | beta shifted +0.10: 299s | p<0.05 in 1.00 of windows, median p 0.00


std | width x0.5: 310s | p<0.05 in 1.00 of windows, median p 0.00


comp | production posterior: 286s | p<0.05 in 0.32 of windows, median p 0.08


comp | beta shifted +0.10: 265s | p<0.05 in 1.00 of windows, median p 0.00


,variant,posterior,frac_p_below_0_05,median_p,min_p,n_windows
6,comp,beta shifted +0.10,1.00,0.00,0.00,25
5,comp,production posterior,0.32,0.08,0.00,25
1,default,beta shifted +0.10,0.92,0.02,0.00,25
0,default,production posterior,0.00,0.62,0.16,25
3,std,beta shifted +0.10,1.00,0.00,0.00,25
2,std,production posterior,0.00,0.54,0.19,25
4,std,width x0.5,1.00,0.00,0.00,25


,window,scenario,default | production posterior,default | beta shifted +0.10,std | production posterior,std | beta shifted +0.10,std | width x0.5,comp | production posterior,comp | beta shifted +0.10
0,0,Sc1,0.69,0.02,0.86,0.0,0.00,0.10,0.0
1,1,Sc1,0.65,0.00,0.46,0.0,0.00,0.00,0.0
2,2,Sc1,0.53,0.00,0.63,0.0,0.00,0.00,0.0
3,3,Sc1,0.82,0.02,0.92,0.0,0.00,0.12,0.0
4,4,Sc1,0.39,0.02,0.94,0.0,0.00,0.07,0.0
5,50,Sc2,0.64,0.03,0.54,0.0,0.00,0.11,0.0
6,51,Sc2,0.62,0.00,0.77,0.0,0.01,0.03,0.0
7,52,Sc2,0.71,0.02,0.82,0.0,0.00,0.14,0.0
8,53,Sc2,0.74,0.00,0.37,0.0,0.00,0.00,0.0
9,54,Sc2,0.84,0.10,0.76,0.0,0.00,0.10,0.0


posterior sd of beta in these windows (mean): 0.044 | a +0.10 shift is 2.3 posterior sd
notebook wall time 35.9 min


## 6. Interpretation

**Result in one line.** With a classifier that has demonstrated power (standardised inputs, section 4) the L-C2ST does **not reject the production posterior in any of the 25 windows** (0 of 25 below 0.05, minimum p = 0.19, median 0.54), while it rejects a posterior that is wrong by 2.3 posterior standard deviations in β, or by a halved width, in **all 25**. The evidence is: no detectable error in the individual posteriors down to about that size. It is not evidence of exactness: a smaller error is not excluded.

**The three runs of this notebook, and why the pre-set default is not the headline.** The pre-set default (raw inputs, sbi's (20, 20) classifier) produced three different answers in three runs with identical seeds and identical data: 13 of 25 windows below 0.05 (median p 0.04), then 0 of 25 (median 0.10), then 0 of 25 (median 0.62). A test whose verdict changes between identical runs cannot be read, and the cause is most likely the unscaled inputs (features from 0.02 mol/L to 400 L/min) making the classifier's training numerically unstable
(multi-threaded floating-point reductions); we did not isolate it. The first run's rejection of 13 windows is therefore an artefact, and the notes that quoted it are corrected here. In the third run the default does detect the shifted posterior (23 of 25 windows, section 4), so it is not powerless, but its null is not reproducible. The standardised variant gave 0 of 25 in both of its runs (median p 0.37 and 0.54): the same verdict.
**The first power check was wrong** (it fed wrong draws to a classifier trained on the correct posterior); the corrected controls in section 4 run the whole test for the wrong posterior.

**The compressed variant is not clean.** With x reduced to the ensemble's own posterior mean and sd (6 inputs) the test flags 8 of 25 windows of the production posterior (p < 0.05; median p 0.08, KS p = 0.000 against U(0,1)), spread over Sc1–Sc4 (2, 2, 1 and 3 of 5 windows) and absent in Sc5, while it flags the wrong posterior in 25 of 25. Either the ensemble is locally miscalibrated as a function of its own mean and sd (a calibration conditional on the posterior summary, which the global SBC and the coverage tables cannot see), or this variant's null is
not well calibrated. We cannot tell which with 25 windows and one calibration set; it is reported as an **open question for the individual-posterior check**, not as a finding about the production posterior. Because the full-x test, which has the same information plus more, finds nothing, we weight the artefact explanation but do not exclude the other.

**What this licenses for the paper.** Evidence for the "reliable" half: SBC and TARP (average calibration), coverage per scenario (notebooks 11, 06), the Cramér–Rao comparison (notebook 08: the posterior is at the bound) and now an individual-posterior test with demonstrated power that finds no error in 25 windows. What it does not: an independent gold standard (the test uses a calibration set from the same simulator, so it cannot detect a simulator misspecification, which is the point of the storyline's silent failures), nor power against errors smaller than about 2 posterior sd,
nor a verdict on the compressed-variant question. The replacement for the NUTS baseline is therefore a set of consistency checks against the model, not a check against the plant.

**Ledger.** Supports: no individual-window miscalibration detected (standardised L-C2ST, 0 of 25; wrong posteriors 25 of 25). Documents: the instability of the sbi default on unscaled inputs; the unresolved compressed-variant result.